# Mission 04: Logging Middleware - 해답 노트북

이 노트북은 네 번째 미션의 완성된 솔루션 코드와 설명입니다.

In [1]:
# 1. 필요한 라이브러리 및 환경 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
# 루트 폴더 기준의 경로 등록
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))  # sys.path.append("app") 대신 "." 등록이 파이썬 패키지 경로 탐색에 안전합니다.
load_dotenv(override=True)

from app.utils.llm import get_llm
from app.utils.context import AgentContext
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from app.tools import web_search

/home/hukim/env_langchain_123/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/hukim/env_langchain_123/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### [미션 1] LoggingMiddleware 구현하기

아래 코드는 `before_agent`, `after_agent`, `wrap_tool_call`을 구현하여 대화 세션 ID별로 로그 파일을 생성해 적재하는 솔루션 코드입니다.

In [2]:
import time
import json
import os
from typing import Any, Dict
from langchain.agents.middleware import AgentMiddleware

class StudentLoggingMiddleware(AgentMiddleware):
    def __init__(self, log_dir="./artifacts/logs"):
        self.log_dir = log_dir
        os.makedirs(self.log_dir, exist_ok=True)
        # 💡 id(runtime) 사전을 사용하지 않으므로 활성 런 관리가 필요 없습니다.

    def _get_session_id(self, runtime, request=None) -> str:
        """컨텍스트 스레드에서 최우선의 session_id(thread_id)를 찾아냅니다."""
        # 1. 🌟 [1번 방식] langchain 컨텍스트 변수에서 RunnableConfig 추출
        try:
            from langchain_core.runnables.config import get_config_from_context
            config = get_config_from_context()
            if config:
                tid = config.get("configurable", {}).get("thread_id")
                if tid:
                    return tid
        except Exception:
            pass

        # 2. ToolRequest의 request.runtime.config에서 추출 (도구 실행 시 fallback)
        if request and hasattr(request, "runtime") and hasattr(request.runtime, "config") and request.runtime.config:
            tid = request.runtime.config.get("configurable", {}).get("thread_id")
            if tid:
                return tid

        # 3. context 객체에 캐싱된 session_id 탐색
        if runtime and hasattr(runtime, "context"):
            tid = getattr(runtime.context, "session_id", None)
            if tid:
                return tid

        return "unknown"

    def before_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        """에이전트 전체 실행의 시작 로깅"""
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None

        start_time = time.time()
        user_query = state.get("messages", [])[-1].content if state.get("messages") else "unknown"
        session_id = self._get_session_id(runtime)
        
        # 💡 [컨텍스트 공유] 스레드 세이프한 runtime.context에 값들을 캐싱합니다.
        if runtime and runtime.context:
            runtime.context.start_time = start_time
            runtime.context.user_query = user_query
            runtime.context.session_id = session_id

        print(f"\n🪵 [LoggingMiddleware] === 에이전트 실행 시작 ===")
        print(f"📥 사용자 질문: {user_query}")
        return None

    def after_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        """에이전트 전체 실행의 완료 및 감사 로그 생성"""
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None

        # context 객체에서 매칭되는 정보를 안전하게 읽어옵니다.
        start_time = getattr(runtime.context, "start_time", None) if runtime and runtime.context else None
        user_query = getattr(runtime.context, "user_query", "unknown") if runtime and runtime.context else "unknown"
        session_id = self._get_session_id(runtime)
        
        duration_ms = int((time.time() - start_time) * 1000) if start_time else 0
        print(f"📤 에이전트 실행 완료 (소요: {duration_ms}ms)")

        # 에이전트 최종 답변 및 전체 대화 궤적 추출
        agent_response = ""
        dialogue_history = []
        messages = state.get("messages", [])
        if messages:
            agent_response = messages[-1].content
            for msg in messages:
                dialogue_history.append({
                    "role": msg.type,
                    "content": str(msg.content)
                })

        # 에이전트 감사 로그 생성 및 파일 적재
        audit_log = {
            "event": "agent_execution",
            "session_id": session_id,
            "query": user_query,
            "response": agent_response,
            "dialogue_history": dialogue_history,
            "latency_ms": duration_ms,
            "status": "SUCCESS" if messages else "FAILED",
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        self._append_log(session_id, audit_log)
        return None

    def wrap_tool_call(self, request, handler):
        """개별 도구(Tool Call) 격발의 시작과 완료를 감싸서 로깅"""
        logging_enabled = getattr(request.runtime.context, "logging_enabled", False) if request.runtime and request.runtime.context else False
        if not logging_enabled:
            return handler(request)

        tool_name = request.tool_call.get("name", "unknown_tool")
        tool_args = request.tool_call.get("args", {})
        start_time = time.time()
        
        print(f"🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: {tool_name}({tool_args})")
        
        # 도구 실행 수행
        response = handler(request)
        
        duration_ms = int((time.time() - start_time) * 1000)
        print(f"🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: {tool_name} (소요: {duration_ms}ms)")
        
        session_id = self._get_session_id(request.runtime, request=request)
        if request.runtime and request.runtime.context:
            request.runtime.context.session_id = session_id
            
        # 💡 [요청 사항 반영] 도구의 반환 데이터(result)도 기록합니다.
        tool_log = {
            "event": "tool_execution",
            "session_id": session_id,
            "tool_name": tool_name,
            "arguments": tool_args,
            "result": str(response),
            "latency_ms": duration_ms,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        self._append_log(session_id, tool_log)
        return response

    def _append_log(self, session_id, log_data):
        """감사 로그 지정 경로에 세션별 JSON라인 파일로 적재"""
        log_file = os.path.join(self.log_dir, f"{session_id}.jsonl")
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_data, ensure_ascii=False) + "\n")


### [미션 2] 미들웨어가 연동된 에이전트 생성 및 검증

작성한 미들웨어를 create_agent에 등록하고 대화를 요청하여 로그 파일이 분리되어 저장되는지 검증합니다.

In [3]:
log_directory = "./artifacts/logs"
thread_id = "session_logging_test_04"
log_filepath = os.path.join(log_directory, f"{thread_id}.jsonl")

if os.path.exists(log_filepath):
    os.remove(log_filepath)

logging_middleware = StudentLoggingMiddleware(log_dir=log_directory)
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)

from app.prompts import CHATBOT_SYSTEM_PROMPT

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=CHATBOT_SYSTEM_PROMPT,
    middleware=[logging_middleware],
    context_schema=AgentContext
)

context_obj = AgentContext(logging_enabled=True)

res = agent.invoke(
    {"messages": [HumanMessage(content="마케팅 기법 중 '바이럴 마케팅'에 대해 web_search 도구를 한 번 써서 간단히 검색하고 핵심 1문장으로 요약해줘.")]},
    config={"configurable": {"thread_id": thread_id}},
    context=context_obj
)
print("최종 답변:", res["messages"][-1].content)


🪵 [LoggingMiddleware] === 에이전트 실행 시작 ===
📥 사용자 질문: 마케팅 기법 중 '바이럴 마케팅'에 대해 web_search 도구를 한 번 써서 간단히 검색하고 핵심 1문장으로 요약해줘.
🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '바이럴 마케팅 정의'})


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 243ms)
📤 에이전트 실행 완료 (소요: 0ms)
최종 답변: [{'type': 'text', 'text': '바이럴 마케팅은 소비자들이 자발적으로 기업의 제품이나 서비스에 대한 소문과 콘텐츠를 바이러스처럼 빠르게 전달하고 확산하도록 유도하는 마케팅 기법입니다.', 'thought_signature': 'AY89a182w3CWEnH1m7uKL3qs+xapyQ49s8D+6Anx7wgWyKEHOZNUJxek0Mjg9sMBfIucxqIKMw9iX9bjUgpc+Sn2EP2N7h7yJ7rSdhoEnuJhGF2adCeaJasY5wBJ6416Ip5iw0qhtB9FTGCgc4aQ0ZOAwym/+C0SEXWG6XqWjsWorS9D1xbwRauBBY4s1qkuxTyC336VHUOyj/ZugOMwhXNL9IWVLY8ASh4WhJn4jsGM/JeMRsB615MPL5kMt9abUNJHOQqrX3RSIabrzXCb3x7WNU4vfqYwZfWxBuLsdEwYQrD8/8J6cnUdBC3pynjwTVnjifjiJDpAzHJ09tW6rY9TcPHfvaNs3OPJNm730siZxgWwtj9r2T3mKhrC7cN2pItECHk7iwlFb0GeFAPr0EjK+tWUGiRjM/v4smN0K0HMSUMYBcLMChsE+KqilxLNnJnjQmh8zzL44qT/Zc9/JU6bGztBIzsgojGloDHj5alJvC0JJnZEQGG9VUvlgQtffDPWmy663afdO7Tm+ZExWQgHxsHISPpaDu8l6+8dtbUJF90IJNJEO95dvW6m4lZh8Nc/w59XvhTQrrWlEdD6lXn3maY='}]


In [4]:
res = agent.invoke(
    {"messages": [HumanMessage(content="이번에는 바이럴 마케팅 관련한 서적을 검색해서 추천해줘.")]},
    config={"configurable": {"thread_id": thread_id}},
    context=context_obj
)
print("최종 답변:", res["messages"][-1].content)


🪵 [LoggingMiddleware] === 에이전트 실행 시작 ===
📥 사용자 질문: 이번에는 바이럴 마케팅 관련한 서적을 검색해서 추천해줘.
🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '바이럴 마케팅 책 추천'})
🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 176ms)


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '컨테이저스 전략적 입소문 책'})
🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 148ms)


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '조나 버거 컨테이저스 책'})
🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 181ms)


...

🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 589ms)
🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '바이럴 마케팅 실무 책 추천'})
🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 123ms)


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '바이럴 마케팅 베스트셀러 도서'})


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 538ms)
📤 에이전트 실행 완료 (소요: 0ms)
최종 답변: [{'type': 'text', 'text': '바이럴 마케팅(Viral Marketing)은 단순히 운에 맡기는 것이 아니라, **사람들이 자발적으로 공유하게 만드는 과학적인 원리와 정교한 설계**가 필요합니다. \n\n바이럴 마케팅의 기초 이론부터 실무 적용, 메시지 기획까지 단계별로 도움이 되는 **최고의 마케팅 서적 5권**을 추천해 드립니다.\n\n---\n\n### 1. 바이럴 마케팅의 절대적 교과서\n#### 📘 《컨테이저스 전략적 입소문》 — 조나 버거 저\n* **한 줄 요약**: "왜 어떤 것은 유행하고, 어떤 것은 묻히는가?"에 대한 과학적 해답.\n* **핵심 내용**: 와튼스쿨 마케팅학 교수인 저자가 수년간의 연구를 통해 밝혀낸 입소문을 만드는 **6가지 법칙(STEPPS)**을 제시합니다.\n  1. **사회적 화폐 (Social Currency)**: 타인에게 멋져 보이고 싶어 공유하는 심리\n  2. **계기 (Triggers)**: 일상 속에서 자연스럽게 제품을 떠올리게 하는 자극\n  3. **감성 (Emotion)**: 마음을 움직여 공유 버튼을 누르게 만드는 감정\n  4. **대중성 (Public)**: 눈에 잘 띄고 모방하기 쉬운 형태\n  5. **실용적 가치 (Practical Value)**: 타인에게 도움을 줄 수 있는 유용한 정보\n  6. **이야기 (Stories)**: 흥미진진한 이야기 속에 자연스럽게 녹아든 브랜드 메시지\n* **추천 대상**: 바이럴이 일어나는 근본적인 인간 심리와 프레임워크를 체계적으로 배우고 싶은 분.\n\n---\n\n### 2. 뇌리에 박히는 메시지 기획법\n#### 📘 《스틱! (Stick!)》 — 칩 히스, 댄 히스 저\n* **한 줄 요약**: 사람들의 기억에 착 달라붙어(Stick) 스스로 퍼져나가는 메시지의 비밀.\

### [미션 3] 생성된 감사 로그 검증

실제로 지정된 경로에 JSON 라인으로 로그들이 정상 저장되었는지 프린트해봅니다.

In [5]:
if os.path.exists(log_filepath):
    print(f"📝 적재된 로그 내용 ({log_filepath}):")
    print("-" * 80)
    with open(log_filepath, "r", encoding="utf-8") as f:
        for line in f:
            print(line.strip())
    print("-" * 80)
else:
    print("❌ 로그 파일이 생성되지 않았습니다.")

📝 적재된 로그 내용 (./artifacts/logs/session_logging_test_04.jsonl):
--------------------------------------------------------------------------------
{"event": "tool_execution", "session_id": "session_logging_test_04", "tool_name": "web_search", "arguments": {"query": "바이럴 마케팅 정의"}, "latency_ms": 243, "timestamp": "2026-08-05T19:48:13Z"}
{"event": "tool_execution", "session_id": "session_logging_test_04", "tool_name": "web_search", "arguments": {"query": "바이럴 마케팅 책 추천"}, "latency_ms": 176, "timestamp": "2026-08-05T19:48:19Z"}
{"event": "tool_execution", "session_id": "session_logging_test_04", "tool_name": "web_search", "arguments": {"query": "컨테이저스 전략적 입소문 책"}, "latency_ms": 148, "timestamp": "2026-08-05T19:48:21Z"}
{"event": "tool_execution", "session_id": "session_logging_test_04", "tool_name": "web_search", "arguments": {"query": "조나 버거 컨테이저스 책"}, "latency_ms": 181, "timestamp": "2026-08-05T19:48:23Z"}
{"event": "tool_execution", "session_id": "session_logging_test_04", "tool_name": "web_